In [12]:
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

In [13]:
import pandas as pd
import numpy as np
from mlxtend.frequent_patterns import apriori, association_rules

DATASET_PATH = "/content/pattern_recognition_preprocessed.csv"

In [14]:
TOP_K_PATTERNS = 10

MIN_SUPPORT = 0.03
MIN_CONFIDENCE = 0.30
MIN_LIFT = 1.05

MAX_RULE_LEN = 4

MIN_CONTEXT_ITEMS = 2
CONF_SIMILARITY_TOL = 0.02

OUTPUT_JSON = "final_rule_database.json"

In [15]:
df = pd.read_csv(DATASET_PATH)
print("Dataset shape:", df.shape)

required_cols = ["gn_pcode", "gn_division"]

for c in required_cols:
    if c not in df.columns:
        raise ValueError(f"Dataset missing required column: {c}")

Dataset shape: (1608, 46)


In [16]:
# AUTO DETECT FEATURE COLUMNS

crime_columns = [c for c in df.columns if c.startswith("crime_")]
location_columns = [c for c in df.columns if c.startswith("loc_")]

excluded_cols = set(["gn_pcode", "gn_division"])
excluded_cols |= set(crime_columns)
excluded_cols |= set(location_columns)

context_columns = []

In [17]:
for c in df.columns:
    if c in excluded_cols:
        continue

    if df[c].dropna().isin([0, 1, True, False]).all():
        context_columns.append(c)

selected_columns = context_columns + location_columns + crime_columns

In [18]:
print("Detected crime columns:", len(crime_columns))
print("Detected location columns:", len(location_columns))
print("Detected context columns:", len(context_columns))

Detected crime columns: 6
Detected location columns: 13
Detected context columns: 13


In [19]:
# HELPER FUNCTIONS

def context_to_crime(rule_row):
    antecedents = rule_row["antecedents"]
    consequents = rule_row["consequents"]

    ant_has_crime = any(item.startswith("crime_") for item in antecedents)
    con_has_crime = any(item.startswith("crime_") for item in consequents)

    return (not ant_has_crime) and con_has_crime

In [20]:
def get_rule_context_items(rule_row):
    ants = set(rule_row["antecedents"])
    context_items = [x for x in ants if (not x.startswith("crime_")) and (not x.startswith("loc_"))]
    return context_items

In [21]:
def contains_required_structure(rule_row):

    ctx = get_rule_context_items(rule_row)

    has_time = any(("time_" in x) for x in ctx)
    has_day = any(("day_" in x) or ("weekday" in x) or ("weekend" in x) for x in ctx)

    return has_time and has_day

In [22]:
def subset_prune_rules(rules_df, conf_tol=0.02):

    rules_df = rules_df.copy()

    keep = [True] * len(rules_df)
    rules_list = rules_df.to_dict("records")

    for i in range(len(rules_list)):

        if not keep[i]:
            continue

        A = rules_list[i]
        A_ant = set(A["antecedents"])
        A_conf = float(A["confidence"])

        for j in range(len(rules_list)):

            if i == j or not keep[j]:
                continue

            B = rules_list[j]
            B_ant = set(B["antecedents"])
            B_conf = float(B["confidence"])

            if A_ant.issubset(B_ant) and len(B_ant) > len(A_ant):

                if abs(A_conf - B_conf) <= conf_tol:
                    keep[j] = False

    return rules_df.loc[keep].copy()